# STAEformer baseline on PEMS-BAY (Kaggle)

The **official STAEformer code and its official PEMS-BAY recipe, unchanged** (repository `XDZhelheim/STAEformer`, which ships `data/PEMSBAY/`). Nothing is tuned.
Only three mechanical edits are made to `train.py`, and all are printed in section 2 (plus, with `FAST_ATTENTION = True`, the attention in `STAEformer.py` is computed by torch's fused `scaled_dot_product_attention`, the same softmax(QK^T/sqrt(d))V, to cut the epoch time):
1. the random seed is read from `SEED` instead of `torch.randint(1000, (1,))`, so a run is reproducible (the official script draws a random seed every run);
2. `verbose=False` is removed from `MultiStepLR` (a deprecated argument that newer torch versions reject; no change in behaviour);
3. the maximum number of epochs is read from `EPOCHS` (set to **30** here; the official value is 300). Everything else, including the learning-rate steps at epochs 10 and 30 and the early stop of 20, stays as in the repository. With 30 epochs the second learning-rate drop (after epoch 30) never happens, so this is a shortened schedule and the numbers will be somewhat worse than the published ones.

Recipe check (paper vs `STAEformer.yaml`, PEMSBAY): input / time-of-day / day-of-week embedding 24, adaptive embedding 80, 3 temporal + 3 spatial layers, 4 heads, feed-forward 256, dropout 0.1, input and output 12 steps, Adam, lr 0.001 with multi-step decay at epochs 10 and 30 (x0.1), weight decay 1e-4, batch 16, masked MAE loss, split 70 / 10 / 20, at most 300 epochs in the official recipe (30 here), early stop after 20 epochs without validation improvement (the paper says 30; the repository's PEMSBAY entry says 20; this notebook follows the repository).

After training, the best checkpoint is evaluated on validation and test at 15 / 30 / 60 minutes (horizons 3 / 6 / 12), in the same format as the other baselines, so the runs can be put side by side.
Published STAEformer PEMS-BAY test MAE for reference: 1.31 / 1.62 / 1.88.

In [ ]:
# ---- settings
SEEDS = (42,)         # seed 42 first, as for the other baselines
EPOCHS = 30           # maximum epochs. The official recipe is 300 (early stop 20); 30 is a shortened budget, say so when reporting
FAST_ATTENTION = True  # replace the hand-written attention by torch's fused scaled_dot_product_attention: the same softmax(QK^T/sqrt(d))V
                       # (outputs differ by ~1e-6 on CPU, checked), but less memory traffic. Without it one epoch took ~19 min on the first run.
DATASET = "PEMSBAY"
REPO_URL = "https://github.com/XDZhelheim/STAEformer.git"
EXPECTED = dict(lr=0.001, weight_decay=0.0001, batch_size=16, max_epochs=300, early_stop=20, milestones=[10, 30], lr_decay_rate=0.1, train_size=0.7, val_size=0.1)
EXPECTED_MODEL = dict(num_nodes=325, in_steps=12, out_steps=12, input_embedding_dim=24, tod_embedding_dim=24, dow_embedding_dim=24,
                      spatial_embedding_dim=0, adaptive_embedding_dim=80, feed_forward_dim=256, num_heads=4, num_layers=3, dropout=0.1)
PUBLISHED_TEST_MAE = (1.31, 1.62, 1.88)

## 1. Environment

In [ ]:
import glob, json, os, re, shutil, subprocess, sys, time
import numpy as np
import torch

DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
print(sys.version.split()[0], "| torch", torch.__version__, "|", DEVICE, torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
assert DEVICE != "cpu", "no GPU: set Accelerator to GPU in the notebook settings"
for mod, pip in (("torchinfo", "torchinfo"), ("yaml", "pyyaml")):
    try:
        __import__(mod)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip], check=True)

## 2. Code, data and the recipe check

In [ ]:
WORK = "/kaggle/working"
SCRATCH = "/tmp/staeformer"
os.makedirs(SCRATCH, exist_ok=True)
CODE = f"{SCRATCH}/code"
shutil.rmtree(CODE, ignore_errors=True)
hits = [p for p in glob.glob("/kaggle/input/**/STAEformer.py", recursive=True)]
if hits:
    shutil.copytree(os.path.dirname(os.path.dirname(hits[0])), CODE)       # the repository was added as an input
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, CODE], check=True)
assert os.path.exists(f"{CODE}/model/train.py") and os.path.exists(f"{CODE}/data/{DATASET}/data.npz"), "STAEformer code or PEMSBAY data not found"
print("code:", CODE, "| data:", os.listdir(f"{CODE}/data/{DATASET}"))

import yaml
cfg = yaml.safe_load(open(f"{CODE}/model/STAEformer.yaml"))[DATASET]
bad = {k: (cfg.get(k), v) for k, v in EXPECTED.items() if cfg.get(k) != v}
bad.update({"model_args." + k: (cfg["model_args"].get(k), v) for k, v in EXPECTED_MODEL.items() if cfg["model_args"].get(k) != v})
assert not bad, f"the repository recipe differs from the recorded one: {bad}"
print("recipe matches the recorded official PEMSBAY recipe")

path = f"{CODE}/model/train.py"
text = open(path).read()
edits = [("seed = torch.randint(1000, (1,)) # set random seed here", 'seed = int(os.environ.get("SEED", 42))', 1),
         ('max_epochs=cfg.get("max_epochs", 200),', 'max_epochs=int(os.environ.get("EPOCHS", cfg.get("max_epochs", 200))),', 1),
         ("        verbose=False,\n", "", 1)]
for old, new, count in edits:
    assert text.count(old) == count, f"train.py: expected {count} x {old!r}, found {text.count(old)}"
    text = text.replace(old, new)
open(path, "w").write(text)
print("patched train.py:", [e[0].strip()[:60] for e in edits])

if FAST_ATTENTION:
    mpath = f"{CODE}/model/STAEformer.py"
    mtext = open(mpath).read()
    a, b = mtext.index("        key = key.transpose("), mtext.index("        out = torch.cat(\n            torch.split(out, batch_size")
    assert "attn_score" in mtext[a:b] and mtext.count("        key = key.transpose(") == 1 and "self.mask" in mtext[a:b]
    assert "mask=False" in mtext and "AttentionLayer(model_dim, num_heads, mask)" in mtext     # the model never builds a masked layer
    mtext = mtext[:a] + "        out = torch.nn.functional.scaled_dot_product_attention(query, key, value)  # = softmax(QK^T / sqrt(head_dim)) V\n" + mtext[b:]
    open(mpath, "w").write(mtext)
    print("STAEformer.py: attention replaced by scaled_dot_product_attention")

## 3. Train with the official recipe
`train.py -d PEMSBAY` writes the log to `logs/` and the best checkpoint to `saved_models/`. Epoch lines are printed as they come (the timestamps show the epoch time).

In [ ]:
EPOCH = re.compile(r"Epoch (\d+)\s+Train Loss = (\S+) Val Loss = (\S+)")
for seed in SEEDS:
    out = f"{WORK}/results_staeformer/staeformer__seed{seed}"
    if os.path.exists(f"{out}/results.json"):
        print("already done: seed", seed)
        continue
    print("--- seed", seed)
    shutil.rmtree(f"{CODE}/logs", ignore_errors=True)
    shutil.rmtree(f"{CODE}/saved_models", ignore_errors=True)
    t0 = time.time()
    lines = []
    with subprocess.Popen([sys.executable, "train.py", "-d", DATASET], cwd=f"{CODE}/model", env=dict(os.environ, SEED=str(seed), EPOCHS=str(EPOCHS), PYTHONUNBUFFERED="1"),
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True) as proc:
        for line in proc.stdout:
            lines.append(line)
            s = line.strip()
            if " Epoch " in s or s.startswith(("Early stopping", "Best at", "Val ", "Train RMSE", "All Steps", "Step 3 ", "Step 6 ", "Step 12 ", "Inference", "Loss:")):
                print(line, end="")
    assert proc.returncode == 0, "train.py failed; last lines:\n" + "".join(lines[-15:])
    os.makedirs(out, exist_ok=True)
    open(f"{out}/train_stdout.txt", "w").write("".join(lines))
    ck = sorted(glob.glob(f"{CODE}/saved_models/*.pt"))[-1]
    shutil.copy(ck, f"{out}/best.pt")
    epochs = [(int(m.group(1)), float(m.group(3))) for m in map(EPOCH.search, lines) if m]
    json.dump({"epochs_run": len(epochs), "best_epoch": min(epochs, key=lambda e: e[1])[0], "training_seconds": time.time() - t0}, open(f"{out}/train_info.json", "w"))

## 4. Evaluate the best checkpoint at 15 / 30 / 60 minutes
Validation and test metrics per horizon with the repository's own masked metrics (zeros are missing and are ignored), on the original speed scale. `results.json` has the same layout as the other baselines.

In [ ]:
sys.path.insert(0, CODE)
os.chdir(f"{CODE}/model")
from lib.data_prepare import get_dataloaders_from_index_data
from lib.metrics import RMSE_MAE_MAPE
from model.STAEformer import STAEformer

H = (3, 6, 12)
_, val_loader, test_loader, SCALER = get_dataloaders_from_index_data(f"{CODE}/data/{DATASET}", tod=cfg["time_of_day"], dow=cfg["day_of_week"], batch_size=cfg["batch_size"])


@torch.no_grad()
def predict(model, loader):
    model.eval()
    ys, outs = [], []
    for x, y in loader:
        o = SCALER.inverse_transform(model(x.to(DEVICE)))
        outs.append(o.cpu().numpy())
        ys.append(y.numpy())
    return np.vstack(ys).squeeze(), np.vstack(outs).squeeze()


for seed in SEEDS:
    out = f"{WORK}/results_staeformer/staeformer__seed{seed}"
    if os.path.exists(f"{out}/results.json"):
        continue
    model = STAEformer(**cfg["model_args"]).to(DEVICE)
    model.load_state_dict(torch.load(f"{out}/best.pt", map_location=DEVICE))
    res = {"run": f"staeformer__seed{seed}", "seed": seed, "parameters": sum(p.numel() for p in model.parameters()), "device": torch.cuda.get_device_name(0),
           **json.load(open(f"{out}/train_info.json")), "epochs": EPOCHS}
    for split, loader in (("validation", val_loader), ("test", test_loader)):
        y, p = predict(model, loader)
        r = {}
        for h in H:
            rmse, mae, mape = RMSE_MAE_MAPE(y[:, h - 1], p[:, h - 1])
            r[str(h)] = {"mae": float(mae), "rmse": float(rmse), "mape": float(mape)}
        r["mean"] = {m: float(np.mean([r[str(h)][m] for h in H])) for m in ("mae", "rmse", "mape")}
        rmse, mae, mape = RMSE_MAE_MAPE(y, p)
        r["all12"] = {"mae": float(mae), "rmse": float(rmse), "mape": float(mape)}
        res[split] = r
        if split == "test":
            np.savez_compressed(f"{out}/test_predictions_h3_6_12.npz", y=y[:, [h - 1 for h in H]].astype(np.float32), pred=p[:, [h - 1 for h in H]].astype(np.float32))
    json.dump(res, open(f"{out}/results.json", "w"), indent=1)
    print("seed", seed, "done")

## 5. Summary

In [ ]:
runs = [json.load(open(f)) for f in sorted(glob.glob(f"{WORK}/results_staeformer/*/results.json"))]
Hs = ("3", "6", "12")
print(f"{'':26s}" + "".join(f"{h:>14s}" for h in ("15 min", "30 min", "60 min", "mean", "all 12")))
for split in ("validation", "test"):
    for metric in ("mae", "rmse", "mape"):
        rows = np.array([[r[split][h][metric] for h in Hs] + [r[split]["mean"][metric], r[split]["all12"][metric]] for r in runs])
        print(f"{split + ' ' + metric.upper():26s}" + "".join(f"{(f'{rows[:, i].mean():.3f}' + (f' +-{rows[:, i].std(ddof=1):.3f}' if len(rows) > 1 else '')):>14s}" for i in range(rows.shape[1])))
print()
for r in runs:
    print(f"{r['run']:26s} best epoch {r['best_epoch']:3d} of {r['epochs_run']:3d} run   {r['training_seconds'] / 60:6.1f} min   {r['parameters']:,} parameters")
t = np.array([[r["test"][h]["mae"] for h in Hs] for r in runs]).mean(0)
print("\ntest MAE 15 / 30 / 60 min:  ours", " / ".join(f"{v:.3f}" for v in t), "  published", " / ".join(f"{v:.2f}" for v in PUBLISHED_TEST_MAE))
print("difference                 ", " / ".join(f"{a - b:+.3f}" for a, b in zip(t, PUBLISHED_TEST_MAE)))